# Rubik's Cube Diffusion Multi-View Probe

This notebook builds directly on the smoother single-view diffusion test.

Its job is to answer one next-step question:

**What kind of second-view pressure gives us the clearest signal for Rubik diffusion: lighter geometric competition, equal geometric competition, or equal semantic competition?**

It now defaults to an official-like setup: Fourier-parameterized source faces, no explicit TV or anchor loss, and score-distillation settings that are much closer to the authors' public Colabs.

You can still switch among a few prompt presets by editing one line:

- geometric circle vs stripes with a lighter scrambled weight
- geometric circle vs stripes with equal weights
- semantic cat vs dog with equal weights

That should tell us whether the next bottleneck is mostly view weighting, prompt choice, or the gap between our Rubik operator and the authors' original arrangement setups.


## Before You Run

- Use a **GPU** runtime.
- The install cell may take a while.
- If this is your first time installing the official dependencies in this runtime, restart the runtime after the install cell finishes, then rerun from the repo bootstrap cell at the top.
- The official code may ask you to `huggingface-cli login` depending on the model/runtime state.
- Change `ACTIVE_EXPERIMENT` in the config cell to switch between geometric vs semantic prompts and light vs equal weighting.
- Change `PARAMETERIZATION_MODE` only if you explicitly want to compare the official-like Fourier setup against the older raster setup.

This notebook is trying to answer one question only:

**How does our Rubik operator behave when we make the learnable image setup look much more like the official Diffusion Illusions notebooks?**


In [ ]:
import os
import platform
import subprocess
import sys
from pathlib import Path

OUR_REPO_URL = 'https://github.com/netalondon/rubiks-diffusion-illusion.git'
OUR_REPO_DIR = Path('/content/rubiks-diffusion-illusion')
OFFICIAL_REPO_URL = 'https://github.com/RyannDaGreat/Diffusion-Illusions.git'
OFFICIAL_REPO_DIR = Path('/content/Diffusion-Illusions')

for repo_url, repo_dir in [
    (OUR_REPO_URL, OUR_REPO_DIR),
    (OFFICIAL_REPO_URL, OFFICIAL_REPO_DIR),
]:
    if repo_dir.exists():
        print(f'Reusing existing repo at {repo_dir}')
        subprocess.check_call(['git', '-C', str(repo_dir), 'pull', '--ff-only'])
    else:
        subprocess.check_call(['git', 'clone', repo_url, str(repo_dir)])

print('Python:', sys.version)
print('Platform:', platform.platform())
print('In Colab runtime:', 'COLAB_RELEASE_TAG' in os.environ)
print('Our repo:', OUR_REPO_DIR)
print('Official repo:', OFFICIAL_REPO_DIR)


In [ ]:
import subprocess
import sys

# The upstream notebook pins several older package versions that do not install cleanly on modern Colab runtimes.
# For this smoke test we only install the libraries the imported modules actually need.
third_party_packages = [
    'diffusers',
    'transformers',
    'rp',
    'easydict',
    'einops',
    'icecream',
    'imageio',
    'opencv-python',
    'pandas',
    'scipy',
    'tensorboard',
    'tensorboardX',
    'tqdm',
    'py3nvml',
]

subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '--upgrade',
    *third_party_packages,
])
subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-r',
    str(OUR_REPO_DIR / 'requirements.txt'),
])

print('Installed a modern Colab-compatible subset of the Diffusion Illusions dependencies.')
print('If this was the first install in this runtime, restart the runtime now, then rerun from the repo bootstrap cell at the top.')


## Imports And Model Setup

The cells below mirror the official notebooks more closely:

- `source.stable_diffusion` comes from the official repo
- the Rubik renderer comes from our repo
- the source-face parameterization is defined locally in this notebook so we can iterate on it safely


In [ ]:
import sys
from pathlib import Path

OUR_REPO_DIR = globals().get('OUR_REPO_DIR', Path('/content/rubiks-diffusion-illusion'))
OFFICIAL_REPO_DIR = globals().get('OFFICIAL_REPO_DIR', Path('/content/Diffusion-Illusions'))

for repo_dir in [OUR_REPO_DIR, OFFICIAL_REPO_DIR]:
    if not repo_dir.exists():
        raise FileNotFoundError(
            f'Missing expected repo directory: {repo_dir}. Run the repo bootstrap cell first.'
        )

if str(OUR_REPO_DIR) not in sys.path:
    sys.path.insert(0, str(OUR_REPO_DIR))
if str(OFFICIAL_REPO_DIR) not in sys.path:
    sys.path.insert(0, str(OFFICIAL_REPO_DIR))

import matplotlib.pyplot as plt
import numpy as np
import rp
import torch
import torch.nn as nn

import source.stable_diffusion as sd
from python_bridge import (
    FACE_FILE_NAMES,
    batch_to_pil_face_dict,
    clone_rendered_to_cpu,
    load_source_faces,
    load_spec,
    render_all_arrangements,
    render_all_arrangements_torch,
    save_rendered_faces,
    stack_source_face_tensors,
    tensor_to_pil,
)
from source.learnable_textures import LearnableImageFourier
from source.stable_diffusion_labels import NegativeLabel

gpu = rp.select_torch_device()
if str(gpu) == 'cpu':
    raise RuntimeError('This smoke test should run on a GPU runtime. Switch Colab to GPU and reconnect.')

if 's' not in globals():
    model_name = 'runwayml/stable-diffusion-v1-5'
    s = sd.StableDiffusion(gpu, model_name)

device = s.device
print('Stable Diffusion device:', device)


In [ ]:
SPEC_PATH = OUR_REPO_DIR / 'public' / 'generated' / 'rubiks-illusion-spec.json'
SOURCE_DIR = OUR_REPO_DIR / 'src' / 'assets' / 'face-art'

spec = load_spec(SPEC_PATH)
face_order = list(spec['primeImages'])
face_to_index = {face_id: index for index, face_id in enumerate(face_order)}

source_faces_pil = load_source_faces(SOURCE_DIR, face_order)
source_batch = stack_source_face_tensors(source_faces_pil, face_order, device=device)

pil_rendered = render_all_arrangements(spec, source_faces_pil)
torch_rendered = render_all_arrangements_torch(spec, source_batch, face_to_index)

max_diff = 0.0
for arrangement_name in pil_rendered:
    for face_id, pil_image in pil_rendered[arrangement_name].items():
        pil_tensor = stack_source_face_tensors({face_id: pil_image}, [face_id], device=device)[0]
        diff = (pil_tensor - torch_rendered[arrangement_name][face_id]).abs().max().item()
        max_diff = max(max_diff, diff)

print('Face order:', face_order)
print('Arrangement names:', list(spec['arrangements'].keys()))
print(f'Max abs difference between PIL and torch Rubik renders: {max_diff:.6f}')


In [ ]:
EXPERIMENTS = {
    'geometric_light': {
        'description': 'Keep the original circle-vs-stripes setup, but keep the scrambled view lighter so we can preserve the main solved structure.',
        'views': [
            {
                'name': 'solved:F',
                'arrangement': 'solved',
                'face': 'F',
                'prompt': 'a minimal graphic poster of a centered soft pink circle on a warm cream background',
                'weight': 1.0,
            },
            {
                'name': 'scrambled:U',
                'arrangement': 'scrambled',
                'face': 'U',
                'prompt': 'a minimal graphic poster with diagonal purple stripes on a pale cream background',
                'weight': 0.35,
            },
        ],
    },
    'geometric_equal': {
        'description': 'Give the circle and stripes equal pressure so we can see whether the secondary view was simply underweighted.',
        'views': [
            {
                'name': 'solved:F',
                'arrangement': 'solved',
                'face': 'F',
                'prompt': 'a minimal graphic poster of a centered soft pink circle on a warm cream background',
                'weight': 1.0,
            },
            {
                'name': 'scrambled:U',
                'arrangement': 'scrambled',
                'face': 'U',
                'prompt': 'a minimal graphic poster with diagonal purple stripes on a pale cream background',
                'weight': 1.0,
            },
        ],
    },
    'animals_equal': {
        'description': 'Try a more semantic solved-plus-scrambled pair where the scrambled target contains three stickers from the solved face.',
        'views': [
            {
                'name': 'solved:R',
                'arrangement': 'solved',
                'face': 'R',
                'prompt': 'a drawing of a cat',
                'weight': 1.0,
            },
            {
                'name': 'scrambled:U',
                'arrangement': 'scrambled',
                'face': 'U',
                'prompt': 'a drawing of a dog',
                'weight': 1.0,
            },
        ],
    },
}

ACTIVE_EXPERIMENT = 'animals_equal'
EXPERIMENT = EXPERIMENTS[ACTIVE_EXPERIMENT]
EXPERIMENT_SLUG = ACTIVE_EXPERIMENT.replace('_', '-')
NEGATIVE_PROMPT = 'blurry, noisy, muddy colors, text, watermark, low quality, collage, multiple animals, cropped face'

train_views = []
for view_config in EXPERIMENT['views']:
    view = dict(view_config)
    view['label'] = NegativeLabel(view['prompt'], NEGATIVE_PROMPT)
    train_views.append(view)

print('Active experiment:', ACTIVE_EXPERIMENT)
print(EXPERIMENT['description'])
for view in train_views:
    print(view['name'], 'weight=', view['weight'], '->', view['prompt'])


In [ ]:
class RubiksLearnableSourceFacesRaster(nn.Module):
    def __init__(self, face_ids, size, stable_diffusion_device, base_rgb=(0.92, 0.90, 0.84), noise_scale=0.03):
        super().__init__()
        self.face_ids = list(face_ids)
        base = torch.tensor(base_rgb, device=stable_diffusion_device, dtype=torch.float32).view(1, 3, 1, 1)
        base = base.repeat(len(self.face_ids), 1, size, size)
        base = (base + noise_scale * torch.randn_like(base)).clamp(0.02, 0.98)
        self.initial_reference = base.detach().clone()
        self.logits = nn.Parameter(torch.logit(base))

    def forward(self):
        return torch.sigmoid(self.logits)


class RubiksLearnableSourceFacesFourier(nn.Module):
    def __init__(self, face_ids, size, hidden_dim=256, num_features=256, scale=10):
        super().__init__()
        self.face_ids = list(face_ids)
        self.initial_reference = None
        self.learnable_images = nn.ModuleList([
            LearnableImageFourier(
                height=size,
                width=size,
                hidden_dim=hidden_dim,
                num_features=num_features,
                scale=scale,
            )
            for _ in self.face_ids
        ])

    def forward(self):
        return torch.stack([image() for image in self.learnable_images])


def total_variation_loss(batch):
    vertical = (batch[:, :, 1:, :] - batch[:, :, :-1, :]).abs().mean()
    horizontal = (batch[:, :, :, 1:] - batch[:, :, :, :-1]).abs().mean()
    return vertical + horizontal


def anchor_loss(batch, reference):
    return (batch - reference).pow(2).mean()


def show_face_dict(face_dict, title, face_ids):
    cols = min(3, max(1, len(face_ids)))
    rows = (len(face_ids) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(3.2 * cols, 3.2 * rows))
    axes = np.asarray(axes, dtype=object).reshape(rows, cols)
    fig.suptitle(title, fontsize=14)

    for axis in axes.flat:
        axis.axis('off')

    for index, face_id in enumerate(face_ids):
        row = index // cols
        col = index % cols
        axis = axes[row, col]
        axis.imshow(np.asarray(face_dict[face_id]))
        axis.set_title(face_id)
        axis.axis('off')

    plt.tight_layout()
    plt.show()


def show_training_views(rendered_views, title):
    items = [
        (
            view['name'],
            rendered_views[view['arrangement']][view['face']],
        )
        for view in train_views
    ]
    fig, axes = plt.subplots(1, len(items), figsize=(4.5 * len(items), 4.5))
    axes = np.asarray(axes, dtype=object).reshape(1, len(items))
    fig.suptitle(title, fontsize=14)

    for axis, (label, image_tensor) in zip(axes[0], items):
        axis.imshow(np.asarray(tensor_to_pil(image_tensor)))
        axis.set_title(label)
        axis.axis('off')

    plt.tight_layout()
    plt.show()


PARAMETERIZATION_MODE = 'official_fourier'

if PARAMETERIZATION_MODE == 'official_fourier':
    learnable_size = 128
    fourier_hidden_dim = 128
    fourier_num_features = 128
    fourier_scale = 10
    learnable_faces = RubiksLearnableSourceFacesFourier(
        face_order,
        learnable_size,
        hidden_dim=fourier_hidden_dim,
        num_features=fourier_num_features,
        scale=fourier_scale,
    ).to(device)
    print('Parameterization mode: official_fourier')
    print('Using LearnableImageFourier to more closely match the official Diffusion Illusions notebooks.')
    print('Fourier config:', {
        'size': learnable_size,
        'hidden_dim': fourier_hidden_dim,
        'num_features': fourier_num_features,
        'scale': fourier_scale,
    })
    initial_title = 'Initial official-like Fourier source faces'
else:
    learnable_size = 96
    learnable_faces = RubiksLearnableSourceFacesRaster(face_order, learnable_size, device).to(device)
    print('Parameterization mode: smooth_raster')
    initial_title = 'Initial smooth raster source faces'


def get_current_state():
    current_source_batch = learnable_faces()
    current_rendered = render_all_arrangements_torch(spec, current_source_batch, face_to_index)
    return current_source_batch, current_rendered


initial_source_batch, initial_rendered = get_current_state()
show_face_dict(
    batch_to_pil_face_dict(initial_source_batch.detach().cpu(), face_order),
    initial_title,
    face_order,
)
show_training_views(clone_rendered_to_cpu(initial_rendered), 'Initial multi-view training render')


In [ ]:
if PARAMETERIZATION_MODE == 'official_fourier':
    NUM_ITER = 5000
    DISPLAY_INTERVAL = 200
    LEARNING_RATE = 1e-4
    GUIDANCE_SCALE = 100
    NOISE_COEF = 0.10
    TV_COEF = 0.0
    ANCHOR_COEF = 0.0
    OPTIMIZER_NAME = 'SGD'

    s.max_step = 990
    s.min_step = 10
    optim = torch.optim.SGD(learnable_faces.parameters(), lr=LEARNING_RATE)
else:
    NUM_ITER = 325
    DISPLAY_INTERVAL = 25
    LEARNING_RATE = 8e-3
    GUIDANCE_SCALE = 80
    NOISE_COEF = 0.11
    TV_COEF = 0.08
    ANCHOR_COEF = 0.005
    OPTIMIZER_NAME = 'Adam'

    s.max_step = 980
    s.min_step = 20
    optim = torch.optim.Adam(learnable_faces.parameters(), lr=LEARNING_RATE)

print('Running diffusion multiview experiment:', ACTIVE_EXPERIMENT)
print(EXPERIMENT['description'])
print('Parameterization mode:', PARAMETERIZATION_MODE)
print('Optimizer:', OPTIMIZER_NAME)
print('Train views:', [(view['name'], view['weight']) for view in train_views])
print('Iterations:', NUM_ITER)
print('Preview interval:', DISPLAY_INTERVAL)
print('Learning rate:', LEARNING_RATE)
print('Guidance scale:', GUIDANCE_SCALE)
print('Noise coefficient:', NOISE_COEF)
print('TV coefficient:', TV_COEF)
print('Anchor coefficient:', ANCHOR_COEF)
print('Min/max step:', (s.min_step, s.max_step))

for iter_num in range(NUM_ITER):
    optim.zero_grad(set_to_none=True)
    current_source_batch, current_rendered = get_current_state()

    sampled_views = list(rp.random_batch(train_views, batch_size=1))
    for view in sampled_views:
        current_view = current_rendered[view['arrangement']][view['face']][None]
        s.train_step(
            view['label'].embedding,
            current_view,
            noise_coef=NOISE_COEF * view['weight'],
            guidance_scale=GUIDANCE_SCALE,
        )

    tv_value = torch.zeros((), device=device)
    anchor_value = torch.zeros((), device=device)
    regularizer_loss = torch.zeros((), device=device)

    if TV_COEF > 0:
        tv_value = total_variation_loss(current_source_batch)
        regularizer_loss = regularizer_loss + TV_COEF * tv_value

    if ANCHOR_COEF > 0 and getattr(learnable_faces, 'initial_reference', None) is not None:
        anchor_value = anchor_loss(current_source_batch, learnable_faces.initial_reference)
        regularizer_loss = regularizer_loss + ANCHOR_COEF * anchor_value

    if regularizer_loss.requires_grad:
        regularizer_loss.backward()

    optim.step()

    if (iter_num + 1) % DISPLAY_INTERVAL == 0 or iter_num == 0:
        print(
            f'Completed iteration {iter_num + 1}/{NUM_ITER} '
            f'| sampled={[view["name"] for view in sampled_views]} '
            f'| tv={tv_value.item():.4f} '
            f'| anchor={anchor_value.item():.4f}'
        )
        _, preview_rendered = get_current_state()
        show_training_views(
            clone_rendered_to_cpu(preview_rendered),
            f'{ACTIVE_EXPERIMENT} preview at iter {iter_num + 1}',
        )


In [ ]:
final_source_batch, final_rendered = get_current_state()
final_source_faces = batch_to_pil_face_dict(final_source_batch.detach().cpu(), face_order)
final_rendered_cpu = clone_rendered_to_cpu(final_rendered)

show_face_dict(final_source_faces, 'Final smooth raster source faces after multi-view probe', face_order)
show_training_views(final_rendered_cpu, 'Final multi-view probe renders')


In [ ]:
from datetime import datetime, timezone
import json
import shutil

OUTPUT_ROOT = OUR_REPO_DIR / 'output' / 'colab-diffusion-multiview-probe' / EXPERIMENT_SLUG
SOURCE_OUTPUT_DIR = OUTPUT_ROOT / 'source-faces'
RENDER_OUTPUT_DIR = OUTPUT_ROOT / 'rendered'
RUNS_ROOT = OUR_REPO_DIR / 'output' / 'colab-runs'
RUN_TIMESTAMP = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H-%M-%SZ')
RUN_ROOT = RUNS_ROOT / f'{RUN_TIMESTAMP}-{EXPERIMENT_SLUG}'
RUN_RESULTS_DIR = RUN_ROOT / 'results'
RUN_SOURCE_DIR = RUN_RESULTS_DIR / 'source-faces'
RUN_RENDER_DIR = RUN_RESULTS_DIR / 'rendered'
RUN_NOTEBOOK_PATH = RUN_ROOT / 'notebook-source.ipynb'
RUN_METADATA_PATH = RUN_ROOT / 'metadata.json'

for directory in [SOURCE_OUTPUT_DIR, RUN_SOURCE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

for face_id, image in final_source_faces.items():
    image.save(SOURCE_OUTPUT_DIR / f'{face_id}.png')
    image.save(RUN_SOURCE_DIR / f'{face_id}.png')

save_rendered_faces(
    {face_id: tensor_to_pil(image) for face_id, image in final_rendered_cpu['solved'].items()},
    RENDER_OUTPUT_DIR / 'solved',
)
save_rendered_faces(
    {face_id: tensor_to_pil(image) for face_id, image in final_rendered_cpu['scrambled'].items()},
    RENDER_OUTPUT_DIR / 'scrambled',
)
save_rendered_faces(
    {face_id: tensor_to_pil(image) for face_id, image in final_rendered_cpu['solved'].items()},
    RUN_RENDER_DIR / 'solved',
)
save_rendered_faces(
    {face_id: tensor_to_pil(image) for face_id, image in final_rendered_cpu['scrambled'].items()},
    RUN_RENDER_DIR / 'scrambled',
)

shutil.copy2(OUR_REPO_DIR / 'experiments' / 'diffusion-multiview-probe' / 'rubiks_colab_diffusion_multiview_probe.ipynb', RUN_NOTEBOOK_PATH)

run_metadata = {
    'experiment': ACTIVE_EXPERIMENT,
    'description': EXPERIMENT['description'],
    'timestamp_utc': RUN_TIMESTAMP,
    'model_name': 'runwayml/stable-diffusion-v1-5',
    'device': str(device),
    'parameterization_mode': PARAMETERIZATION_MODE,
    'negative_prompt': NEGATIVE_PROMPT,
    'train_views': [
        {
            'name': view['name'],
            'arrangement': view['arrangement'],
            'face': view['face'],
            'prompt': view['prompt'],
            'weight': view['weight'],
        }
        for view in train_views
    ],
    'hyperparameters': {
        'num_iter': NUM_ITER,
        'display_interval': DISPLAY_INTERVAL,
        'learning_rate': LEARNING_RATE,
        'optimizer_name': OPTIMIZER_NAME,
        'guidance_scale': GUIDANCE_SCALE,
        'noise_coef': NOISE_COEF,
        'tv_coef': TV_COEF,
        'anchor_coef': ANCHOR_COEF,
        'min_step': s.min_step,
        'max_step': s.max_step,
    },
    'paths': {
        'canonical_output_root': str(OUTPUT_ROOT),
        'run_root': str(RUN_ROOT),
        'run_notebook_source': str(RUN_NOTEBOOK_PATH),
    },
    'notes': [
        'This run snapshot stores outputs, prompts, weights, and a clean copy of the notebook source.',
        'Executed cell outputs from the VS Code or Colab editor are not embedded automatically by the runtime.',
    ],
}
RUN_METADATA_PATH.write_text(json.dumps(run_metadata, indent=2) + '\\n')

print('Saved experiment:', ACTIVE_EXPERIMENT)
print('Saved canonical source faces to:', SOURCE_OUTPUT_DIR)
print('Saved canonical rendered solved faces to:', RENDER_OUTPUT_DIR / 'solved')
print('Saved canonical rendered scrambled faces to:', RENDER_OUTPUT_DIR / 'scrambled')
print('Saved run snapshot to:', RUN_ROOT)
print('Saved run metadata to:', RUN_METADATA_PATH)


In [ ]:
import shutil

ARCHIVE_BASE = RUN_ROOT.parent / f'{RUN_ROOT.name}-export'
ARCHIVE_PATH = shutil.make_archive(str(ARCHIVE_BASE), 'zip', root_dir=RUN_ROOT)
print('Created export archive:', ARCHIVE_PATH)

try:
    from google.colab import files
    files.download(ARCHIVE_PATH)
    print('Started download via google.colab.files.download')
except Exception as error:
    print('Automatic download was not available in this runtime:', repr(error))
    print('The zip file is still available inside Colab at:', ARCHIVE_PATH)
    print('You can download it manually from the Colab file browser if needed.')


## What Success Looks Like

This notebook is a success if all of these are true:

- Stable Diffusion loads successfully
- the smooth multi-view loop runs without shape or device errors
- the active preset produces a more informative tradeoff than the lighter geometric baseline alone
- semantic presets tell us whether cat/dog-style imagery survives Rubik competition better than simple posters
- each run writes a timestamped snapshot under `output/colab-runs/...` with metadata, images, and a notebook source copy
- the snapshot can be exported as a zip when you want to bring it back from Colab

Once that works, the next step is to decide whether we should tune the two-view tradeoff further or move toward a broader Rubik objective with more faces.
